### Section E

###E1

In [0]:
dbutils.widgets.text("catalog", "fk_ce_2026_07_1378")
catalog = dbutils.widgets.get("catalog")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold")

In [0]:
from pyspark.sql import functions as F

# E1: gold.dim_date — one row per day, 2026-03-01 to 2026-03-31
dim_date = (
    spark.range(31)
    .withColumn("calendar_date", F.date_add(F.lit("2026-03-01"), F.col("id").cast("int")))
    .withColumn("date_key", F.date_format(F.col("calendar_date"), "yyyyMMdd").cast("int"))
    .withColumn("day_name", F.date_format(F.col("calendar_date"), "EEEE"))
    .withColumn("iso_week", F.weekofyear(F.col("calendar_date")))
    .withColumn("is_weekend", F.date_format(F.col("calendar_date"), "EEEE").isin(["Saturday", "Sunday"]))
    .select("date_key", "calendar_date", "day_name", "iso_week", "is_weekend")
)

dim_date.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.gold.dim_date")
print(f"dim_date: {dim_date.count()} rows")

In [0]:
from pyspark.sql import functions as F

# E1: gold.dim_store — all columns of silver.stores
spark.read.table(f"{catalog}.silver.stores") \
    .write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.gold.dim_store")
print(f"dim_store: {spark.table(f'{catalog}.gold.dim_store').count()} rows")

# E1: gold.dim_product — all columns of silver.products except tags and last_changed_snapshot_date
spark.read.table(f"{catalog}.silver.products") \
    .drop("tags", "last_changed_snapshot_date") \
    .write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.gold.dim_product")
print(f"dim_product: {spark.table(f'{catalog}.gold.dim_product').count()} rows")

In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F

# E1: gold.dim_customer — customer_sk (BIGINT, unique, never NULL) + all columns of silver.customers
silver_customers = spark.read.table(f"{catalog}.silver.customers")

# Generate deterministic surrogate key ordered by customer_id, effective_from
w = Window.orderBy("customer_id", "effective_from")
dim_customer = (
    silver_customers
    .withColumn("customer_sk", F.row_number().over(w).cast("long"))
    .select("customer_sk", *silver_customers.columns)
)

dim_customer.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.gold.dim_customer")
print(f"dim_customer: {dim_customer.count()} rows")

###E2

In [0]:
from pyspark.sql import functions as F

# E2: gold.fact_orders — one row per silver.orders row
fact_orders = spark.sql(f"""
WITH line_agg AS (
    SELECT
        so.order_id,
        CAST(COUNT(li) AS INT) AS line_count,
        CAST(SUM(COALESCE(li.quantity, 0)) AS INT) AS units,
        CAST(SUM(COALESCE(li.quantity * li.unit_price, 0)) AS DECIMAL(12,2)) AS gross_amount,
        CAST(SUM(COALESCE(li.discount_amount, 0)) AS DECIMAL(12,2)) AS discount_amount,
        CAST(SUM(COALESCE(li.quantity * li.unit_price - li.discount_amount, 0)) AS DECIMAL(12,2)) AS net_item_amount
    FROM {catalog}.silver.orders so
    LATERAL VIEW OUTER EXPLODE(so.items) AS li
    GROUP BY so.order_id
),
bronze_delivery AS (
    SELECT DISTINCT
        UPPER(TRIM(order_id)) AS order_id,
        CAST(delivery.delivery_fee AS DECIMAL(10,2)) AS delivery_fee,
        CAST(delivery.promised_mins AS INT) AS promised_mins,
        CAST(delivery.actual_mins AS INT) AS actual_mins,
        try_to_timestamp(updated_at) AS updated_at
    FROM {catalog}.bronze.orders
)
SELECT
    so.order_id,
    CAST(date_format(so.order_date, 'yyyyMMdd') AS INT) AS order_date_key,
    dc.customer_sk,
    so.customer_id,
    so.store_id,
    so.order_status,
    so.payment_method,
    so.channel,
    COALESCE(la.line_count, 0) AS line_count,
    COALESCE(la.units, 0) AS units,
    CAST(la.gross_amount AS DECIMAL(12,2)) AS gross_amount,
    CAST(la.discount_amount AS DECIMAL(12,2)) AS discount_amount,
    CAST(la.net_item_amount AS DECIMAL(12,2)) AS net_item_amount,
    bd.delivery_fee,
    CAST(COALESCE(la.net_item_amount, 0) + COALESCE(bd.delivery_fee, 0) AS DECIMAL(12,2)) AS order_total,
    bd.promised_mins,
    bd.actual_mins,
    CASE
        WHEN so.order_status = 'DELIVERED' THEN
            CASE WHEN bd.actual_mins <= bd.promised_mins THEN TRUE ELSE FALSE END
        ELSE NULL
    END AS is_on_time
FROM {catalog}.silver.orders so
LEFT JOIN line_agg la ON so.order_id = la.order_id
LEFT JOIN bronze_delivery bd ON so.order_id = bd.order_id AND so.updated_at = bd.updated_at
LEFT JOIN {catalog}.gold.dim_customer dc
    ON so.customer_id = dc.customer_id
    AND dc.effective_from <= so.order_date
    AND (dc.effective_to IS NULL OR dc.effective_to > so.order_date)
""")

fact_orders.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.gold.fact_orders")

# Verify: one row per silver.orders, no NULL customer_sk
so_count = spark.table(f"{catalog}.silver.orders").count()
fo_count = fact_orders.count()
null_sk = fact_orders.filter("customer_sk IS NULL").count()
print(f"silver.orders: {so_count} rows")
print(f"fact_orders:    {fo_count} rows  (match: {so_count == fo_count})")
print(f"NULL customer_sk: {null_sk}")
display(fact_orders.limit(5))

In [0]:
from pyspark.sql import functions as F

# E2: gold.fact_order_line — one row per silver.order_items row (line-grain fact table)
fact_order_line = spark.sql(f"""
SELECT
    oi.order_id,
    oi.line_number,
    oi.product_id,
    oi.quantity,
    oi.unit_price,
    oi.discount_amount,
    oi.gross_amount,
    oi.net_amount,
    CAST(date_format(so.order_date, 'yyyyMMdd') AS INT) AS order_date_key,
    dc.customer_sk,
    so.customer_id,
    so.store_id,
    so.order_status,
    so.payment_method,
    so.channel,
    oi.updated_at
FROM {catalog}.silver.order_items oi
JOIN {catalog}.silver.orders so
  ON oi.order_id = so.order_id AND oi.updated_at = so.updated_at
LEFT JOIN {catalog}.gold.dim_customer dc
    ON so.customer_id = dc.customer_id
    AND dc.effective_from <= so.order_date
    AND (dc.effective_to IS NULL OR dc.effective_to > so.order_date)
""")

fact_order_line.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.gold.fact_order_line")

# Verify: one row per silver.order_items
oi_count = spark.table(f"{catalog}.silver.order_items").count()
fol_count = fact_order_line.count()
null_sk = fact_order_line.filter("customer_sk IS NULL").count()
print(f"silver.order_items: {oi_count} rows")
print(f"fact_order_line:    {fol_count} rows  (match: {oi_count == fol_count})")
print(f"NULL customer_sk: {null_sk}")
display(fact_order_line.limit(5))

###E3

In [0]:
from pyspark.sql import functions as F

# E3: gold.agg_daily_store_sales — one row per store per day (including zero-activity store-days)
agg_daily_store_sales = spark.sql(f"""
WITH bounds AS (
    SELECT
        MIN(to_date(CAST(order_date_key AS STRING), 'yyyyMMdd')) AS min_date,
        MAX(to_date(CAST(order_date_key AS STRING), 'yyyyMMdd')) AS max_date
    FROM {catalog}.gold.fact_orders
),
date_grid AS (
    SELECT explode(sequence(b.min_date, b.max_date, INTERVAL 1 DAY)) AS order_date
    FROM bounds b
),
store_grid AS (
    SELECT DISTINCT store_id, city
    FROM {catalog}.gold.dim_store
),
grid AS (
    SELECT s.store_id, s.city, d.order_date
    FROM store_grid s CROSS JOIN date_grid d
),
agg AS (
    SELECT
        to_date(CAST(order_date_key AS STRING), 'yyyyMMdd') AS order_date,
        store_id,
        CAST(SUM(CASE WHEN order_status = 'DELIVERED' THEN 1 ELSE 0 END) AS INT) AS delivered_orders,
        CAST(SUM(CASE WHEN order_status = 'CANCELLED' THEN 1 ELSE 0 END) AS INT) AS cancelled_orders,
        CAST(SUM(CASE WHEN order_status = 'DELIVERED' THEN COALESCE(units, 0) ELSE 0 END) AS INT) AS units_sold,
        CAST(SUM(CASE WHEN order_status = 'DELIVERED' THEN COALESCE(gross_amount, 0) ELSE 0 END) AS DECIMAL(12,2)) AS gross_amount,
        CAST(SUM(CASE WHEN order_status = 'DELIVERED' THEN COALESCE(discount_amount, 0) ELSE 0 END) AS DECIMAL(12,2)) AS discount_amount,
        CAST(SUM(CASE WHEN order_status = 'DELIVERED' THEN COALESCE(net_item_amount, 0) ELSE 0 END) AS DECIMAL(12,2)) AS net_item_revenue,
        CAST(SUM(CASE WHEN order_status = 'DELIVERED' THEN COALESCE(delivery_fee, 0) ELSE 0 END) AS DECIMAL(12,2)) AS delivery_fee_revenue,
        CAST(SUM(CASE WHEN is_on_time = TRUE THEN 1 ELSE 0 END) AS INT) AS on_time_orders
    FROM {catalog}.gold.fact_orders
    GROUP BY 1, 2
)
SELECT
    g.order_date,
    g.store_id,
    g.city,
    CAST(COALESCE(a.delivered_orders, 0) AS INT) AS delivered_orders,
    CAST(COALESCE(a.cancelled_orders, 0) AS INT) AS cancelled_orders,
    CAST(COALESCE(a.units_sold, 0) AS INT) AS units_sold,
    CAST(COALESCE(a.gross_amount, 0) AS DECIMAL(12,2)) AS gross_amount,
    CAST(COALESCE(a.discount_amount, 0) AS DECIMAL(12,2)) AS discount_amount,
    CAST(COALESCE(a.net_item_revenue, 0) AS DECIMAL(12,2)) AS net_item_revenue,
    CAST(COALESCE(a.delivery_fee_revenue, 0) AS DECIMAL(12,2)) AS delivery_fee_revenue,
    CAST(COALESCE(a.on_time_orders, 0) AS INT) AS on_time_orders,
    CAST(
        CASE
            WHEN COALESCE(a.delivered_orders, 0) > 0
            THEN ROUND(COALESCE(a.on_time_orders, 0) * 100.0 / COALESCE(a.delivered_orders, 0), 2)
            ELSE NULL
        END AS DECIMAL(5,2)
    ) AS on_time_rate_pct
FROM grid g
LEFT JOIN agg a ON g.order_date = a.order_date AND g.store_id = a.store_id
ORDER BY g.store_id, g.order_date
""")

agg_daily_store_sales.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.gold.agg_daily_store_sales")

# Verify: grid coverage, zero-activity rows, NULL safety
store_count = spark.table(f"{catalog}.gold.dim_store").select("store_id").distinct().count()
date_count = spark.table(f"{catalog}.gold.fact_orders").agg(
    F.countDistinct(F.to_date(F.col("order_date_key").cast("string"), "yyyyMMdd"))
).collect()[0][0]
total_rows = agg_daily_store_sales.count()
zero_rows = agg_daily_store_sales.filter("delivered_orders = 0 AND cancelled_orders = 0 AND units_sold = 0").count()
null_rate = agg_daily_store_sales.filter("delivered_orders > 0 AND on_time_rate_pct IS NULL").count()
null_counts = agg_daily_store_sales.filter("delivered_orders IS NULL OR units_sold IS NULL OR gross_amount IS NULL").count()

print(f"agg_daily_store_sales: {total_rows} rows")
print(f"Expected: {store_count} stores × {date_count} dates = {store_count * date_count}")
print(f"Zero-activity store-days: {zero_rows}")
print(f"NULL count/amount rows: {null_counts}")
print(f"Rows with delivered > 0 but NULL on_time_rate_pct: {null_rate}")
display(agg_daily_store_sales.limit(10))

###E4

In [0]:
# E4: gold.v_kpi_summary — single-row KPI view for the whole period
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.gold.v_kpi_summary AS
WITH base AS (
    SELECT
        SUM(CASE WHEN order_status = 'DELIVERED' THEN 1 ELSE 0 END) AS delivered_orders,
        SUM(CASE WHEN order_status = 'CANCELLED' THEN 1 ELSE 0 END) AS cancelled_orders,
        SUM(CASE WHEN order_status = 'DELIVERED' THEN COALESCE(net_item_amount, 0) ELSE 0 END) AS net_item_revenue,
        SUM(CASE WHEN order_status = 'DELIVERED' AND is_on_time = TRUE THEN 1 ELSE 0 END) AS on_time_orders
    FROM {catalog}.gold.fact_orders
),
active AS (
    SELECT COUNT(DISTINCT customer_id) AS active_customers
    FROM {catalog}.gold.fact_orders
    WHERE order_status = 'DELIVERED'
),
repeat AS (
    SELECT COUNT(*) AS repeat_customers
    FROM (
        SELECT customer_id
        FROM {catalog}.gold.fact_orders
        WHERE order_status = 'DELIVERED'
        GROUP BY customer_id
        HAVING COUNT(*) >= 2
    )
)
SELECT
    CAST(delivered_orders AS INT) AS delivered_orders,
    CAST(cancelled_orders AS INT) AS cancelled_orders,
    CAST(net_item_revenue AS DECIMAL(18,2)) AS net_item_revenue,
    CAST(net_item_revenue / NULLIF(delivered_orders, 0) AS DECIMAL(18,2)) AS avg_order_value,
    CAST(100.0 * on_time_orders / NULLIF(delivered_orders, 0) AS DECIMAL(5,2)) AS on_time_rate_pct,
    CAST(100.0 * cancelled_orders / NULLIF(delivered_orders + cancelled_orders, 0) AS DECIMAL(5,2)) AS cancellation_rate_pct,
    CAST(active_customers AS INT) AS active_customers,
    CAST(100.0 * repeat_customers / NULLIF(active_customers, 0) AS DECIMAL(5,2)) AS repeat_customer_rate_pct
FROM base, active, repeat
""")

# Verify: exactly one row
v_kpi = spark.table(f"{catalog}.gold.v_kpi_summary")
print(f"v_kpi_summary row count: {v_kpi.count()}")
display(v_kpi)